# Notebook 4: LSTM-SNP with Fuzzy Output Layer

**Dataset**: S&P 500

## Description
This notebook implements a **fuzzy output layer** for the LSTM-SNP model. The internal 
LSTM-SNP structure is kept completely intact. The final Dense(1) layer is replaced with a 
fuzzy inference system that takes the RNN hidden state h(t) and produces the prediction 
through Takagi-Sugeno rules.

The fuzzy output layer aggregates h(t) into 2 summary features via mean-pooling, then applies 
4 rules with fixed Gaussian membership functions and trainable consequent parameters.

## Theory: Fuzzy Output Layer

### LSTM-SNP Cell (Unchanged)
All internal LSTM-SNP equations remain exactly as in the baseline.

### Fuzzy Output Computation
The Dense(1) output layer is replaced with fuzzy inference:

**Input**: $h(t) \in \mathbb{R}^{units}$ from the RNN

**Feature Aggregation**: Mean-pooling into 2 summary statistics:
- $s_1 = \text{mean}(h_{1:units/2})$
- $s_2 = \text{mean}(h_{units/2+1:units})$

**Membership Functions** (Fixed Gaussian):
- $\mu_{low}(s) = \exp\left(-\frac{(s-(-1))^2}{2 \cdot 0.5^2}\right)$
- $\mu_{high}(s) = \exp\left(-\frac{(s-(+1))^2}{2 \cdot 0.5^2}\right)$

**4 Takagi-Sugeno Rules** (trainable consequents):
$y_i = a_i s_1 + b_i s_2 + c_i$

**Defuzzification**: $\hat{y} = \frac{\sum_i w_i y_i}{\sum_i w_i}$

## Model Architecture & Implementation

In [1]:
# ============================================================
# ALL IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from math import sqrt
from scipy import stats
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PyTorch version: 2.11.0+cu128
NumPy version: 2.5.1
Using device: cuda


### LSTM-SNP Cell

In [2]:
#with noise


class LSTMSNPCell(nn.Module):
    """
    LSTM-SNP Cell: gates r, c, o (hard sigmoid) and generated spikes a (tanh).
    u(t) = r(t)*u(t-1) - c(t)*a(t)
    h(t) = o(t)*a(t)
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.kernel = nn.Linear(input_size, hidden_size * 4, bias=False)
        self.recurrent_kernel = nn.Linear(hidden_size, hidden_size * 4, bias=False)
        self.bias = nn.Parameter(torch.zeros(hidden_size * 4))

        nn.init.xavier_uniform_(self.kernel.weight)
        nn.init.orthogonal_(self.recurrent_kernel.weight)

    def hard_sigmoid(self, x):
        return torch.clamp(0.2 * x + 0.5, 0.0, 1.0)

    def forward(self, x, u_tm1):
        z = self.kernel(x) + self.recurrent_kernel(u_tm1) + self.bias
        z0, z1, z2, z3 = z.chunk(4, dim=-1)

        r = self.hard_sigmoid(z0)   # reset
        c = self.hard_sigmoid(z1)   # consumption
        o = self.hard_sigmoid(z2)   # output/generation
        a = torch.tanh(z3)          # generated spikes

        u = r * u_tm1 - c * a
        h = o * a
        return h, u

### Fuzzy Output Layer

In [3]:
# ============================================================
# Fuzzy Output Layer (PyTorch)
# Replaces Dense(1) with fuzzy inference.
#
# Takes RNN output h(t) ∈ R^units, produces scalar prediction.
# Aggregates h(t) into 2 summary features via mean-pooling:
#   s1 = mean(h[:units//2])
#   s2 = mean(h[units//2:])
#
# Then applies 4 Takagi-Sugeno rules.
# Fixed Gaussian MFs. Trainable consequent parameters.
# ============================================================

import math
import torch
import torch.nn as nn

class FuzzyOutputLayer(nn.Module):
    """
    Fuzzy output layer: replaces Dense(1).
    Input: h(t) from RNN cell (shape: [batch, units])
    Output: scalar prediction (shape: [batch, 1])
    """
    def __init__(self, units_in):
        super().__init__()
        self.units_in = units_in
        self.mu_low = -1.0
        self.mu_high = 1.0
        self.sigma = 0.5

        # 4 rules, each with 3 consequent params (a, b, c)
        # y_i = a_i * s1 + b_i * s2 + c_i
        self.rule_a = nn.Parameter(torch.empty(4))
        self.rule_b = nn.Parameter(torch.empty(4))
        self.rule_c = nn.Parameter(torch.zeros(4))

        limit = math.sqrt(6.0 / 8.0)  # glorot_uniform equivalent for a length-4 vector
        nn.init.uniform_(self.rule_a, -limit, limit)
        nn.init.uniform_(self.rule_b, -limit, limit)

    def _gaussian_mf(self, x, center):
        return torch.exp(-torch.square(x - center) / (2.0 * self.sigma ** 2))

    def forward(self, inputs):
        half = self.units_in // 2
        # Aggregate to 2 summary features
        s1 = inputs[:, :half].mean(dim=-1, keepdim=True)   # [batch, 1]
        s2 = inputs[:, half:].mean(dim=-1, keepdim=True)   # [batch, 1]

        # Membership degrees
        mu_low_s1 = self._gaussian_mf(s1, self.mu_low)
        mu_high_s1 = self._gaussian_mf(s1, self.mu_high)
        mu_low_s2 = self._gaussian_mf(s2, self.mu_low)
        mu_high_s2 = self._gaussian_mf(s2, self.mu_high)

        # Rule weights
        w1 = mu_low_s1 * mu_low_s2
        w2 = mu_low_s1 * mu_high_s2
        w3 = mu_high_s1 * mu_low_s2
        w4 = mu_high_s1 * mu_high_s2

        # Consequent outputs
        y1 = self.rule_a[0] * s1 + self.rule_b[0] * s2 + self.rule_c[0]
        y2 = self.rule_a[1] * s1 + self.rule_b[1] * s2 + self.rule_c[1]
        y3 = self.rule_a[2] * s1 + self.rule_b[2] * s2 + self.rule_c[2]
        y4 = self.rule_a[3] * s1 + self.rule_b[3] * s2 + self.rule_c[3]

        # Defuzzification
        numerator = w1 * y1 + w2 * y2 + w3 * y3 + w4 * y4
        denominator = w1 + w2 + w3 + w4 + 1e-8

        return numerator / denominator

### Build Model

In [4]:
# ============================================================
# Model Construction: LSTM-SNP with Fuzzy Output Layer (PyTorch)
# LSTMSNPCell is UNMODIFIED.
# Dense(1) is replaced by FuzzyOutputLayer.
# Stateful across calls (like Keras stateful=True) via self.u;
# reset explicitly with reset_states(batch_size, device).
# ============================================================

class LSTMSNPFuzzyModel(nn.Module):
    def __init__(self, input_dim, units):
        super().__init__()
        self.units = units
        self.cell = LSTMSNPCell(input_dim, units)
        self.fuzzy_out = FuzzyOutputLayer(units_in=units)
        self.u = None

    def reset_states(self, batch_size, device):
        self.u = torch.zeros(batch_size, self.units, device=device)

    def detach_states(self):
        # Truncate BPTT: keep the state's value, drop its graph history
        if self.u is not None:
            self.u = self.u.detach()

    def forward(self, x):
        # x: [batch, seq_len, input_dim]
        batch_size, seq_len, _ = x.shape
        if self.u is None or self.u.size(0) != batch_size:
            self.reset_states(batch_size, x.device)

        h = None
        for t in range(seq_len):
            h, self.u = self.cell(x[:, t, :], self.u)
        return self.fuzzy_out(h)


def build_model(input_dim, units):
    return LSTMSNPFuzzyModel(input_dim, units)

In [5]:
# Quick model check
model = build_model(input_dim=1, units=8).to(device)
model.reset_states(1, device)
print(model)
print(f"Total trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

LSTMSNPFuzzyModel(
  (cell): LSTMSNPCell(
    (kernel): Linear(in_features=1, out_features=32, bias=False)
    (recurrent_kernel): Linear(in_features=8, out_features=32, bias=False)
  )
  (fuzzy_out): FuzzyOutputLayer()
)
Total trainable params: 332


## Data Pipeline — S&P 500

In [6]:
# ============================================================
# 1. Load Time Series Data
# ============================================================
series = pd.read_csv(
    r'C:\Users\paulp\OneDrive\Desktop\fuzzy_LSTM\dataset\sp500.csv',
    header=0,
    parse_dates=[0],
    index_col=0
)
raw_values = series.values.flatten()
print(f"Data shape: {raw_values.shape}")
print(f"First 5 values: {raw_values[:5]}")

Data shape: (251,)
First 5 values: [2038.199951 2013.780029 2011.709961 1985.319946 1945.969971]


In [7]:
# ============================================================
# 2. First-Order Differencing
# ============================================================

def difference(dataset, interval=1):
    diff = []
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return np.array(diff)

In [8]:
# ============================================================
# 3. Convert to Supervised Learning Format (lag=1)
# ============================================================

def timeseries_to_supervised(data, lag=1):
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag+1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values

In [9]:
# ============================================================
# 3b. Apply differencing + supervised-format conversion
# (evaluate_on_test() adds raw_values[t] back to predictions,
# which only makes sense if the model is trained on first
# differences -- so that step has to happen here.)
# ============================================================
diff_values = difference(raw_values, interval=1)
supervised = timeseries_to_supervised(diff_values, lag=5)
print(f"Differenced series: {diff_values.shape}, supervised: {supervised.shape}")

Differenced series: (250,), supervised: (250, 6)


In [10]:
# ============================================================
# 4. Train-Validation-Test Split (Last 60 points as Test)
# ============================================================
n = len(supervised)
test_size = 60
train_val = supervised[:n - test_size]
test = supervised[n - test_size:]

val_size = int(len(train_val) * 0.1)
train = train_val[:-val_size]
val = train_val[-val_size:]

print(f"Train: {train.shape}, Val: {val.shape}, Test: {test.shape}")

# ============================================================
# 5. Feature Scaling
# ============================================================
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(train)  # fit only on train to avoid leakage
train_scaled = scaler.transform(train)
val_scaled = scaler.transform(val)
test_scaled = scaler.transform(test)

Train: (171, 6), Val: (19, 6), Test: (60, 6)


In [11]:
# ============================================================
# 6. Reshape for RNN Input
# ============================================================

X_train, y_train = train_scaled[:, 0:-1], train_scaled[:, -1]
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))

X_test, y_test = test_scaled[:, 0:-1], test_scaled[:, -1]
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (171, 1, 5), y_train shape: (171,)


In [12]:
#Gaussian noise injection (Eq. 29–30)
ref_std = X_train.std()
print(f"Reference std(x) used for noise scaling (train inputs): {ref_std:.6f}")

NOISE_LEVELS = [0.005, 0.05, 0.10, 0.15]  # 0.5%, 5%, 10%, 15%

def add_gaussian_noise(X, noise_level, ref_std, seed=None):
    """
    x_noisy(t) = x(t) + eps(t),  eps(t) ~ N(0, sigma_eps^2),  sigma_eps = noise_level * std(x)
    Applied to input features X only — never to targets/labels.
    """
    rng = np.random.default_rng(seed)
    sigma_eps = noise_level * ref_std
    eps = rng.normal(loc=0.0, scale=sigma_eps, size=X.shape)
    return X + eps

Reference std(x) used for noise scaling (train inputs): 0.235458


In [13]:
#Evaluation function

def evaluate_on_test(model, X_test_eval, train, train_scaled, raw_values, scaler, device):
    """
    Warms up the model's recurrent state on clean training data, then runs
    single-step-ahead prediction on X_test_eval (clean or Gaussian-noise-corrupted
    input features). Test targets are always the clean, ground-truth values.
    """
    model.eval()
    model.reset_states(1, device)

    with torch.no_grad():
        for i in range(len(train_scaled)):
            X_raw = train_scaled[i, 0:-1]
            X_input = torch.tensor(X_raw, dtype=torch.float32).view(1, 1, len(X_raw)).to(device)
            model(X_input)

        predictions = []
        for i in range(len(X_test_eval)):
            X = X_test_eval[i]
            X_input = torch.tensor(X, dtype=torch.float32).view(1, 1, len(X)).to(device)
            yhat = model(X_input).item()

            new_row = [x for x in X] + [yhat]
            array = np.array(new_row).reshape(1, len(new_row))
            inverted = scaler.inverse_transform(array)[0, -1]
            inverted = inverted + raw_values[len(train) + i]
            predictions.append(inverted)

    actual = raw_values[-len(X_test_eval):]
    mse = mean_squared_error(actual, predictions)
    rmse = sqrt(mse)
    meanV = np.mean(actual)
    dominator = np.linalg.norm(np.array(actual) - meanV, 2)   # FIX: was predictions - meanV
    nmse = mse / np.power(dominator, 2)
    return predictions, rmse, mse, nmse

## Training Loop

In [14]:
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CHECKPOINT_DIR = "checkpoints_sp500"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def _ckpt_path(run):
    return os.path.join(CHECKPOINT_DIR, f"run_{run}.pt")

def _summary_path():
    return os.path.join(CHECKPOINT_DIR, "summary.pt")

all_rmse = []
all_mse = []
all_nmse = []
all_predictions = []
all_losses = []
all_models = []  # keep every trained model for later noise-robustness eval

print(f"\n--- [PyTorch] RUNNING ON {device} ---\n")
print(f"Checkpoints will be read/written under: {os.path.abspath(CHECKPOINT_DIR)}\n")

N_RUNS = 60
val_X = val_scaled[:, 0:-1]
PATIENCE = 10
INPUT_DIM = X_train.shape[-1]  # derived from data (== lag), not hardcoded

summary_path = _summary_path()
if os.path.exists(summary_path):
    print("Found completed checkpoint summary -> loading, skipping training.")
    saved = torch.load(summary_path, map_location=device, weights_only=False)

    all_models = []
    for sd in saved['model_state_dicts']:
        m = build_model(input_dim=INPUT_DIM, units=8).to(device)
        m.load_state_dict(sd)
        m.eval()
        all_models.append(m)

    all_rmse = saved['rmse']
    all_mse = saved['mse']
    all_nmse = saved['nmse']
    all_predictions = saved['predictions']
    all_losses = saved['losses']
    print(f"Loaded {len(all_models)} models from checkpoint.")

else:
    for run in range(N_RUNS):
        run_ckpt_path = _ckpt_path(run)
        if os.path.exists(run_ckpt_path):
            print(f'\n===== RUN {run + 1}/{N_RUNS} (resumed from checkpoint) =====')
            ckpt = torch.load(run_ckpt_path, map_location=device, weights_only=False)

            model = build_model(input_dim=INPUT_DIM, units=8).to(device)
            model.load_state_dict(ckpt['model_state_dict'])
            model.eval()

            all_losses.append(ckpt['run_losses'])
            all_models.append(model)
            all_rmse.append(ckpt['rmse'])
            all_mse.append(ckpt['mse'])
            all_nmse.append(ckpt['nmse'])
            all_predictions.append(ckpt['predictions'])
            print(f"Loaded run {run + 1} -- RMSE: {ckpt['rmse']:.6f}, MSE: {ckpt['mse']:.6f}, NMSE: {ckpt['nmse']:.10f} "
                  f"(best val RMSE: {ckpt['best_val_rmse']:.6f})")
            continue

        print(f'\n===== RUN {run + 1}/{N_RUNS} =====')

        np.random.seed(run)
        torch.manual_seed(run)
        model = build_model(input_dim=INPUT_DIM, units=8).to(device)
        # No LSTM-SNP-style gate-bias init here -- nn.LSTM's own default
        # initialization is used; PureLSTM has no .cell/.hidden_size to slice into.

        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()
        run_losses = []

        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
        n_samples = X_train_t.size(0)

        best_val_rmse = float('inf')
        best_state = None
        patience_ctr = 0

        for epoch in range(100):
            model.train()
            model.reset_states(1, device)
            epoch_loss = 0.0

            for i in range(n_samples):
                x_i = X_train_t[i:i + 1]
                y_i = y_train_t[i:i + 1]

                optimizer.zero_grad()
                pred = model(x_i)
                loss = criterion(pred.squeeze(-1), y_i)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                model.detach_states()  # truncate BPTT between steps
                epoch_loss += loss.item()

            avg_loss = epoch_loss / n_samples
            run_losses.append(avg_loss)

            _, val_rmse, _, _ = evaluate_on_test(model, val_X, train, train_scaled, raw_values, scaler, device)
            print(f"Epoch {epoch + 1}/100 — train loss: {avg_loss:.6f}  val RMSE: {val_rmse:.6f}")

            if val_rmse < best_val_rmse:
                best_val_rmse = val_rmse
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    print(f"Early stopping at epoch {epoch + 1} (best val RMSE: {best_val_rmse:.6f})")
                    break

        model.load_state_dict(best_state)
        model.reset_states(1, device)

        all_losses.append(run_losses)
        all_models.append(model)
        print(f'Training complete for run {run + 1} (best val RMSE: {best_val_rmse:.6f})')

        predictions, rmse, mse, nmse = evaluate_on_test(
            model, X_test, train, train_scaled, raw_values, scaler, device
        )

        all_rmse.append(rmse)
        all_mse.append(mse)
        all_nmse.append(nmse)
        all_predictions.append(predictions)

        print(f'Run {run + 1} — RMSE: {rmse:.6f}, MSE: {mse:.6f}, NMSE: {nmse:.10f}')

        torch.save({
            'model_state_dict': model.state_dict(),
            'run_losses': run_losses,
            'rmse': rmse,
            'mse': mse,
            'nmse': nmse,
            'predictions': predictions,
            'best_val_rmse': best_val_rmse,
        }, run_ckpt_path)

    torch.save({
        'rmse': all_rmse,
        'mse': all_mse,
        'nmse': all_nmse,
        'predictions': all_predictions,
        'losses': all_losses,
        'model_state_dicts': [m.state_dict() for m in all_models],
    }, summary_path)


--- [PyTorch] RUNNING ON cuda ---

Checkpoints will be read/written under: c:\Users\paulp\OneDrive\Desktop\fuzzy_LSTM\Type_4\checkpoints_sp500

Found completed checkpoint summary -> loading, skipping training.
Loaded 60 models from checkpoint.


## Results

In [15]:
#Noise-robustness sweep (averaged over multiple noise draws, with explicit clean baseline)

N_NOISE_DRAWS = 10  # average over multiple ε(t) realizations per model, per level

# Re-evaluate the clean (0% noise) baseline through evaluate_on_test, so it
# uses the exact same evaluation path as the noisy conditions below.
clean_rmse, clean_mse, clean_nmse = [], [], []
for model in all_models:
    _, rmse, mse, nmse = evaluate_on_test(model, X_test, train, train_scaled, raw_values, scaler, device)
    clean_rmse.append(rmse)
    clean_mse.append(mse)
    clean_nmse.append(nmse)

print(f"[{'no noise':>10}]  "
      f"RMSE: {np.mean(clean_rmse):.6f} ± {np.std(clean_rmse):.6f}  |  "
      f"MSE: {np.mean(clean_mse):.6f} ± {np.std(clean_mse):.6f}  |  "
      f"NMSE: {np.mean(clean_nmse):.10f} ± {np.std(clean_nmse):.10f}")

noise_robustness = {}  # noise_level -> dict of per-model RMSE/MSE/NMSE (averaged over draws)

for noise_level in NOISE_LEVELS:
    level_rmse, level_mse, level_nmse = [], [], []

    for run, model in enumerate(all_models):
        draw_rmse, draw_mse, draw_nmse = [], [], []
        for draw in range(N_NOISE_DRAWS):
            seed = hash((noise_level, run, draw)) % (2 ** 32)
            X_test_noisy = add_gaussian_noise(X_test, noise_level, ref_std, seed=seed)
            _, rmse, mse, nmse = evaluate_on_test(
                model, X_test_noisy, train, train_scaled, raw_values, scaler, device
            )
            draw_rmse.append(rmse)
            draw_mse.append(mse)
            draw_nmse.append(nmse)

        level_rmse.append(np.mean(draw_rmse))  # this model's average over noise draws
        level_mse.append(np.mean(draw_mse))
        level_nmse.append(np.mean(draw_nmse))

    noise_robustness[noise_level] = {'rmse': level_rmse, 'mse': level_mse, 'nmse': level_nmse}
    print(f"[{noise_level * 100:5.1f}% noise]  "
          f"RMSE: {np.mean(level_rmse):.6f} ± {np.std(level_rmse):.6f}  |  "
          f"MSE: {np.mean(level_mse):.6f} ± {np.std(level_mse):.6f}  |  "
          f"NMSE: {np.mean(level_nmse):.10f} ± {np.std(level_nmse):.10f}")

[  no noise]  RMSE: 58.397606 ± 0.424988  |  MSE: 3410.460973 ± 49.098542  |  NMSE: 0.0202247191 ± 0.0002911642
[  0.5% noise]  RMSE: 58.397748 ± 0.424586  |  MSE: 3410.477193 ± 49.053163  |  NMSE: 0.0202248152 ± 0.0002908951
[  5.0% noise]  RMSE: 58.397056 ± 0.422039  |  MSE: 3410.394837 ± 48.773422  |  NMSE: 0.0202243268 ± 0.0002892362
[ 10.0% noise]  RMSE: 58.396102 ± 0.421849  |  MSE: 3410.285161 ± 48.736662  |  NMSE: 0.0202236764 ± 0.0002890182
[ 15.0% noise]  RMSE: 58.388397 ± 0.422957  |  MSE: 3409.389256 ± 48.848930  |  NMSE: 0.0202183636 ± 0.0002896840
